In [1]:
# Importing libraries
import numpy as np
import pandas as pd
from ga import Ga
from cross_validation_class import CrossValidation
from yrandomization import YRandomization
from lno import LNO
from filter import variance_cut,correlation_cut
import lj_cut as lj
from validate_yr_lno import validate
import os
import ipywidgets as widgets
from IPython.display import display
from draw_widgets import DrawWidgets
from ipyfilechooser import FileChooser
dw = DrawWidgets()

## Run the cell below in order to generate the buttons to load the matrices 

In [2]:
fileX = widgets.FileUpload(
    accept='',  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description = "X matrix",
)
display(fileX)
filey = widgets.FileUpload(cccc
    accept='',  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    description = "y vector",
)
display(filey)

FileUpload(value={}, description='X matrix')

FileUpload(value={}, description='y vector')

## Run the cell below in order to adequate the matrices to the correct format used for calculations

In [4]:
df = dw.mountMatrix(fileX)
y = dw.mountyvector(filey)

## Run the cell below to choose the GA parameters. The values pre-selected are the optimal or default values.

In [5]:
var_cut_widget = dw.drawFloatSlider(value=0.1,
                              min=0,
                              max=1,
                              description='',
                              width="150pt"
                             )
# display(var_cut_widget)
display(widgets.HBox([widgets.Label('Variance cut:'), var_cut_widget]))

corr_cut_widget = dw.drawFloatSlider(value=0.3,
                              min=0,
                              max=1,
                              description='',
                              width="150pt"
                             )
# display(corr_cut_widget)
display(widgets.HBox([widgets.Label('Correlation cut:'), corr_cut_widget]))

nLVModel_widget = dw.drawIntSlider(value=int(len(df)/5),
                              min=1,
                              max=min(df.shape),
                              description='',
                              width="150pt"
                             )
# display(nLVModel_widget)
display(widgets.HBox([widgets.Label('Maximum number of latent variables for the model:'), nLVModel_widget]))

min_size_widget = dw.drawIntSlider(value=4,
                              min=1,
                              max=df.shape[1],
                              description='',
                              width="150pt"
                             )
# display(nLVModel_widget)
display(widgets.HBox([widgets.Label('Minimum number of variables to be selected:'), min_size_widget]))

max_size_widget = dw.drawIntSlider(value=15,
                              min=1,
                              max=df.shape[1],
                              description='',
                              width="150pt"
                             )
# display(nLVModel_widget)
display(widgets.HBox([widgets.Label('Maximum number of variables to be selected:'), max_size_widget]))

size_population_widget = dw.drawIntSlider(value=200,
                              min=1,
                              max=1000,
                              description='',
                              width="150pt"
                             )
# display(nLVModel_widget)
display(widgets.HBox([widgets.Label('Size of the population in each generation of the GA:'), size_population_widget]))

mig_rate_widget = dw.drawFloatSlider(value=0.2,
                              min=0,
                              max=1,
                              description='',
                              width="150pt"
                             )
# display(corr_cut_widget)
display(widgets.HBox([widgets.Label('Migration rate in GA:'), mig_rate_widget]))

cxpb_widget = dw.drawFloatSlider(value=0.5,
                              min=0,
                              max=1,
                              description='',
                              width="150pt"
                             )
# display(corr_cut_widget)
display(widgets.HBox([widgets.Label('Crossover rate of population:'), cxpb_widget]))

mutpb_widget = dw.drawFloatSlider(value=0.2,
                              min=0,
                              max=1,
                              description='',
                              width="150pt"
                             )
# display(corr_cut_widget)
display(widgets.HBox([widgets.Label('Mutation rate of the population:'), mutpb_widget]))


ngen_widget = dw.drawIntSlider(value=200,
                              min=1,
                              max=1000,
                              description='',
                              width="150pt"
                             )
# display(nLVModel_widget)
display(widgets.HBox([widgets.Label('Number of generations in GA:'), ngen_widget]))

yr_crit_widget = dw.drawFloatSlider(value=0.3,
                              min=0,
                              max=1,
                              description='',
                              width="150pt"
                             )
# display(yr_crit_widget)
display(widgets.HBox([widgets.Label('Criterion to aprove a model in y-randomization:'), yr_crit_widget]))

lno_crit_widget = dw.drawFloatSlider(value=0.1,
                              min=0,
                              max=1,
                              description='',
                              width="150pt"
                             )
# display(lno_crit_widget)
display(widgets.HBox([widgets.Label('Criterion to aprove a model in leave-N-out test:'), lno_crit_widget]))

autoscale_widget = widgets.RadioButtons(
    options=['Yes', 'No'],
#     value='pineapple',
    description='Autoscale?',
    disabled=False
)
display(autoscale_widget)

MIF_transform_widget = widgets.RadioButtons(
    options=['Yes', 'No'],
#     value='pineapple',
    description='',
    disabled=False,
    width="350pt"
)
# display(MIF_transform_widget)
widgets.HBox([widgets.Label('Transform MIF values?'), MIF_transform_widget])

RadioButtons(description='Autoscale?', options=('Yes', 'No'), value='Yes')

## Run the cell below to choose the output files names

In [6]:
print("Choose the directory and type the desired filename for the matrix with the selected variables for best model")
out_matrix_widget = FileChooser(os.getcwd())
display(out_matrix_widget)

print("Choose the directory and type the desired filename to save the cross-validation results for the best models")
out_cv_widget = FileChooser(os.getcwd())
display(out_cv_widget)

print("Choose the directory and type the desired filename to save the best models according to GA selection")
out_models_widget = FileChooser(os.getcwd())
display(out_models_widget)

Choose the directory and type the desired filename for the matrix with the selected variables for best model


FileChooser(path='/home/jpam/anaconda3/envs/QSARmodeling/src', filename='', show_hidden='False')

Choose the directory and type the desired filename to save the cross-validation results for the best models


FileChooser(path='/home/jpam/anaconda3/envs/QSARmodeling/src', filename='', show_hidden='False')

Choose the directory and type the desired filename to save the best models according to GA selection


FileChooser(path='/home/jpam/anaconda3/envs/QSARmodeling/src', filename='', show_hidden='False')

In [8]:
# Open configuration file in order to look for the matrices and the parameters to run
# GA and cross-validation
var_cut = var_cut_widget.value
corr_cut = corr_cut_widget.value
nLVModel = nLVModel_widget.value
min_size = min_size_widget.value
max_size = max_size_widget.value
size_population = size_population_widget.value
mig_rate = mig_rate_widget.value
cxpb = cxpb_widget.value
mutpb = mutpb_widget.value
ngen = ngen_widget.value
yr_crit = yr_crit_widget.value
lno_crit = lno_crit_widget.value
autoscale  = autoscale_widget.value == "Yes"
out_matrix = out_matrix_widget.selected
out_cv = out_cv_widget.selected
var_sel_file = out_models_widget.selected
# Filtering the matrix according to the options in configuration file
dfX = lj.transform(df) if MIF_transform_widget.value == "Yes" else df
print("Dimensions of the original matrix")
print(dfX.shape)
indVar = variance_cut(dfX.values,var_cut)
dfVar = dfX.loc[:,dfX.columns[indVar]]
print("Dimensions of the matrix after variance cut")
print(dfVar.shape)
indCorr = correlation_cut(dfVar.values,y,corr_cut)
dfCorr = dfVar.loc[:,dfVar.columns[indCorr]]
print("Dimensions of the matrix after correlation cut")
print(dfCorr.shape)
X = dfCorr.values
if nLVModel == None:
    nLVModel = dfCorr.shape[0]
ga = Ga(X,y,nLVModel, autoscale, min_size, max_size, size_population, mig_rate, cxpb, mutpb, ngen)
ga.run()
ga.savePop(var_sel_file)
Q2 = ga.Q2
Q2 = [Q2[i][0] for i,_ in enumerate(Q2)]
var_sel = validate(X,y,ga.pop_selected,Q2,yr_cut=yr_crit,lno_cut=lno_crit)
if var_sel != []:
    dfSel = dfCorr.loc[:,dfCorr.columns[var_sel]]
    dfSel.to_csv(out_matrix,sep=';')
    cv = CrossValidation(dfSel.values,y)
    cv.saveParameters(out_cv)        
else:
    print("y-randomization or LNO failed!")

Dimensions of the original matrix
(55, 23)
Dimensions of the matrix after variance cut
(55, 22)
Dimensions of the matrix after correlation cut
(55, 21)
Generation 1 of 200
Generation 2 of 200
Generation 3 of 200
Generation 4 of 200
Generation 5 of 200
Generation 6 of 200
Generation 7 of 200
Generation 8 of 200
Generation 9 of 200
Generation 10 of 200
Generation 11 of 200
Generation 12 of 200
Generation 13 of 200
Generation 14 of 200
Generation 15 of 200
Generation 16 of 200
Generation 17 of 200
Generation 18 of 200
Generation 19 of 200
Generation 20 of 200
Generation 21 of 200
Generation 22 of 200
Generation 23 of 200
Generation 24 of 200
Generation 25 of 200
Generation 26 of 200
Generation 27 of 200
Generation 28 of 200
Generation 29 of 200
Generation 30 of 200
Generation 31 of 200
Generation 32 of 200
Generation 33 of 200
Generation 34 of 200
Generation 35 of 200
Generation 36 of 200
Generation 37 of 200
Generation 38 of 200
Generation 39 of 200
Generation 40 of 200
Generation 41 of 